[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_07/MFM_ch7_tailshape/MFM_ch7_tailshape.ipynb)

# MFM_ch7_tailshape

Time-varying extreme tail shape (D'Innocenzo, Lucas, Schwaab and Zhang, 2024, Journal of Business & Economic Statistics, https://doi.org/10.1080/07350015.2023.2260439) on the daily S&P 500 losses of the course, 2000-2026: dynamic 90% quantile threshold estimated by the pinball loss, score-driven generalised Pareto distribution (GPD) tail shape and scale estimated by maximum likelihood, filtered Value-at-Risk (VaR) 1% and Expected Shortfall (ES) 2.5%; comparison by AIC with a static GPD and with a constant-shape model; the same estimation on 2000-2015 only.

*Modelling Financial Markets (MFM), Chapter 7: Value-at-Risk and Expected Shortfall. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [1]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_7'
    os.makedirs(OUT_DIR, exist_ok=True)
    os.chdir(OUT_DIR)  # relative chart and CSV outputs persist in Drive
print('Save your outputs to:', os.path.abspath(OUT_DIR))


Save your outputs to: /content


In [2]:
# Colab: numba is preinstalled; without it the filters run in plain Python (slower)
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import optimize, stats
try:
    from numba import njit
except ImportError:
    def njit(*a, **k):
        return (lambda f: f) if not a or not callable(a[0]) else a[0]
import warnings
warnings.filterwarnings('ignore')


## Style

In [3]:
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'savefig.facecolor': 'none',
                     'savefig.transparent': True, 'axes.grid': False, 'font.family': 'sans-serif',
                     'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'], 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.linewidth': 0.6, 'legend.frameon': False})

# chapter colours
MainBlue, IDAred, Forest, Orange, Teal, Navy, BandBlue = ('#1A3A6E', '#CD0000', '#2E7D32', '#E67E22', '#17A2B8', '#1F2A44',
                                                    '#C5D2E8')


## Data

In [4]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# use the local copy of the course data when the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# name -> (symbol, label, type, start date)
MARKETS = {
    'sp500':  ('GSPC.INDX',    'S&P 500',                  'index',  '2000-01-01'),
    'bet':    ('BET',          'BET',                      'index',  '2000-01-01'),
    'btc':    ('BTC-USD.CC',   'Bitcoin',                  'crypto', '2014-09-17'),
    'eurron': ('REF:EUR',      'EUR/RON (reference rate)', 'fx',     '2005-07-01'),
    'gold':   ('XAUUSD.FOREX', 'Gold (XAU/USD)',           'fx',     '2000-01-01'),
}
# portfolio assets (prices aligned on common days)
ASSETS = {
    'SPY': ('SPY.US', 'adjusted_close'), 'TLT': ('TLT.US', 'adjusted_close'), 'GLD': ('GLD.US', 'adjusted_close'),
    'BTC': ('BTC-USD.CC', 'close'),
    'TLV': ('TLV.RO', 'adjusted_close'), 'SNP': ('SNP.RO', 'adjusted_close'), 'BRD': ('BRD.RO', 'adjusted_close'),
    'TGN': ('TGN.RO', 'adjusted_close'), 'SNG': ('SNG.RO', 'adjusted_close'), 'SNN': ('SNN.RO', 'adjusted_close'),
    'EL': ('EL.RO', 'adjusted_close'), 'FP': ('FP.RO', 'adjusted_close'), 'TEL': ('TEL.RO', 'adjusted_close'),
}
LABELS = {k: v[1] for k, v in MARKETS.items()}

_CACHE = {}


def read_market(symbol):
    """Read the daily price series of one symbol (local copy of the course data, else from GitHub)."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official RON reference rate (annual BNR XML archives)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END):
    """Daily price, cleaned according to the chapter conventions."""
    symbol, _, kind, start0 = MARKETS[name]
    start = start or start0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    d = read_market(symbol).loc[start:end]
    d = d[d['close'] > 0].dropna(subset=['close'])
    if kind != 'crypto':
        d = d[d.index.dayofweek < 5]          # no weekend quotes
    s = d['close']
    if kind == 'index' and 'volume' in d:
        # drop holidays filled with the previous price: unchanged close and zero/missing volume
        s = s[~((s.diff() == 0) & (d['volume'].fillna(0) <= 0))]
    return s.rename(name)


def log_returns(name, start=None, end=END):
    """Log returns on the series' own calendar."""
    return np.log(load_close(name, start, end)).diff().dropna().rename(name)


## Model

Loss $y_t = -100\,\ln(P_t/P_{t-1})$. Threshold $\tau_{t+1} = (1-b_\tau)\hat q_{0.9} + a_\tau(\mathbf 1\{y_t>\tau_t\}-0.1) + b_\tau\tau_t$; exceedances $x_t = y_t-\tau_t>0$ follow a GPD$(\xi_t,\delta_t)$ with $f_t=(\ln\xi_t,\ln\delta_t)'$ and $f_{t+1}=\omega+As_t+Bf_t$, $s_t$ the score scaled by the square-root inverse Fisher information ($s_t=0$ without an exceedance).

In [5]:
KAPPA = 0.90                    # threshold = dynamic 90% quantile of the loss (paper: 1 - kappa = 10%)
G_VAR, G_ES = 0.99, 0.975       # VaR 1%, ES 2.5%
EST_END, OOS_START = '2015-12-31', '2016-01-01'

# chapter colours (generate_all_charts.py)

plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'savefig.facecolor': 'none',
                     'savefig.transparent': True, 'axes.grid': False, 'font.family': 'sans-serif',
                     'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'], 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.linewidth': 0.6, 'legend.frameon': False})


# ----------------------------------------------------------------------------------------------------------------
# 1. dynamic threshold, eq. (9), estimated by the check loss, eq. (17)
@njit(cache=False)
def _tau_path(y, a, b, q, kappa):
    tau = np.empty(len(y))
    tau[0] = q
    for t in range(len(y) - 1):
        hit = 1.0 if y[t] > tau[t] else 0.0
        tau[t + 1] = (1 - b) * q + a * (hit - (1 - kappa)) + b * tau[t]
    return tau


def tau_path(y, a, b, q):
    return _tau_path(np.asarray(y, float), float(a), float(b), float(q), KAPPA)


def check_loss(y, tau):
    u = y - tau
    return float(np.mean(u * (KAPPA - (u < 0))))


def fit_tau(y):
    q = float(np.quantile(y, KAPPA))
    best = None
    for a0 in (0.05, 0.1, 0.2, 0.3, 0.5):
        for b0 in (0.9, 0.95, 0.98, 0.99, 0.995):
            obj = lambda p: (check_loss(y, tau_path(y, p[0], p[1], q))   # noqa: E731
                             if p[0] > 0 and 0 < p[1] < 1 else 1e9)
            r = optimize.minimize(obj, [a0, b0], method='Nelder-Mead',
                                  options={'xatol': 1e-6, 'fatol': 1e-10, 'maxiter': 4000})
            if best is None or r.fun < best.fun:
                best = r
    return dict(a=float(best.x[0]), b=float(best.x[1]), q=q, loss=float(best.fun))


# ----------------------------------------------------------------------------------------------------------------
# 2. score-driven GPD, eqs. (3)-(6), likelihood (15)
@njit(cache=False)
def _gpd_filter(x, fx, fd, ax, ad, bx, bd):
    n = len(x)
    xi, de = np.empty(n), np.empty(n)
    ox, od = (1 - bx) * fx, (1 - bd) * fd
    f1, f2 = fx, fd
    ll = 0.0
    for t in range(n):
        f1 = min(max(f1, -6.0), 1.5)
        f2 = min(max(f2, -6.0), 4.0)
        k, d = np.exp(f1), np.exp(f2)
        xi[t], de[t] = k, d
        s1, s2 = 0.0, 0.0
        if x[t] > 0:
            z = 1 + k * x[t] / d
            ll += -np.log(d) - (1 + 1 / k) * np.log(z)
            g1 = np.log(z) / k - (1 + 1 / k) * k * x[t] / (d + k * x[t])
            g2 = (x[t] - d) / (d + k * x[t])
            s1 = (1 + 1 / k) * g1 - g2            # L_t' grad_t, eq. (6)
            s2 = np.sqrt(1 + 2 * k) * g2
        f1 = ox + ax * s1 + bx * f1
        f2 = od + ad * s2 + bd * f2
    return xi, de, ll


def gpd_filter(x, theta):
    """x: exceedances (<= 0 where there is none); theta = (fbar_xi, fbar_delta, a_xi, a_delta, b_xi, b_delta),
    omega = (I - B) fbar, f_1 = fbar. Returns xi_t, delta_t (known at t - 1) and the log-likelihood (eq. 15)."""
    return _gpd_filter(np.asarray(x, float), *[float(v) for v in theta])


def fit_gpd_dynamic(x, n_starts=200, seed=42):
    """Maximum likelihood (eq. 15): Nelder-Mead from the static GPD fit and from n_starts random starting points."""
    ex = x[x > 0]
    c, _, sc = stats.genpareto.fit(ex, floc=0)
    neg = lambda p: (1e10 if (p[2] < 0 or p[3] < 0 or not (0 < p[4] < 0.99999) or not (0 < p[5] < 0.99999))   # noqa: E731
                     else -gpd_filter(x, p)[2])
    rng = np.random.default_rng(seed)
    starts = [[np.log(max(c, 0.05)), np.log(sc), 0.02, 0.1, 0.999, 0.99]]
    starts += [[rng.uniform(-4, -0.7), rng.uniform(-1.5, 0.5), rng.uniform(0, 0.3), rng.uniform(0, 0.3),
                1 - 10 ** rng.uniform(-4, -1), 1 - 10 ** rng.uniform(-4, -1)] for _ in range(n_starts)]
    best = None
    for p0 in starts:
        r = optimize.minimize(neg, p0, method='Nelder-Mead',
                              options={'xatol': 1e-8, 'fatol': 1e-9, 'maxiter': 20000, 'maxfev': 20000})
        if best is None or r.fun < best.fun:
            best = r
    return [float(v) for v in best.x], float(-best.fun), dict(xi=float(c), delta=float(sc),
                                                              loglik=float(stats.genpareto.logpdf(ex, c, 0, sc).sum()))


def fit_constant_shape(x, n_starts=100, seed=7):
    """Restricted model: xi constant (a_xi = 0), delta_t score-driven; parameters (fbar_xi, fbar_delta, a_delta, b_delta)."""
    ex = x[x > 0]
    c, _, sc = stats.genpareto.fit(ex, floc=0)
    neg = lambda p: (1e10 if (p[2] < 0 or not (0 < p[3] < 0.99999))   # noqa: E731
                     else -gpd_filter(x, [p[0], p[1], 0.0, p[2], 0.5, p[3]])[2])
    rng = np.random.default_rng(seed)
    starts = [[np.log(max(c, 0.05)), np.log(sc), 0.1, 0.99]]
    starts += [[rng.uniform(-4, -0.7), rng.uniform(-1.5, 0.5), rng.uniform(0, 0.3), 1 - 10 ** rng.uniform(-4, -1)]
               for _ in range(n_starts)]
    best = None
    for p0 in starts:
        r = optimize.minimize(neg, p0, method='Nelder-Mead',
                              options={'xatol': 1e-8, 'fatol': 1e-9, 'maxiter': 20000, 'maxfev': 20000})
        if best is None or r.fun < best.fun:
            best = r
    return [float(v) for v in best.x], float(-best.fun)


def risk(tau, xi, de, share, gamma):
    """VaR and ES at confidence gamma of the loss (Section 2.5 of the paper)."""
    var = tau + de / xi * (((1 - gamma) / share) ** (-xi) - 1)
    es = var / (1 - xi) + (de - xi * tau) / (1 - xi)
    return var, es


def run(y, tp, theta):
    tau = tau_path(y, tp['a'], tp['b'], tp['q'])
    x = y - tau
    xi, de, ll = gpd_filter(x, theta)
    cnt = np.concatenate([[0.0], np.cumsum(x > 0)[:-1]])          # exceedances up to t - 1
    tt = np.arange(len(y), dtype=float)
    share = np.where(cnt > 0, cnt / np.maximum(tt, 1), 1 - KAPPA)  # t*/t
    v1, e1 = risk(tau, xi, de, share, G_VAR)
    v25, e25 = risk(tau, xi, de, share, G_ES)
    return dict(tau=tau, x=x, xi=xi, delta=de, var1=v1, es1=e1, var25=v25, es25=e25, ll=ll, share=share)


EVENTS = {'2001-09-17': '9/11 reopening', '2008-09-29': 'TARP vote', '2010-05-06': 'Flash crash',
          '2011-08-08': 'US downgrade', '2016-06-24': 'Brexit vote', '2018-02-05': 'Volmageddon',
          '2020-03-16': 'COVID-19', '2024-08-05': 'Yen carry unwind'}
NAMES = ['fbar_xi', 'fbar_delta', 'a_xi', 'a_delta', 'b_xi', 'b_delta']


def main():
    r = log_returns('sp500')
    y_s = (-100 * r).astype(float)
    dates = y_s.index
    y = y_s.values

    # full sample, as in the paper's Section 4.1 (parameters on all data; xi_t, delta_t, tau_t use only the past)
    tp = fit_tau(y)
    th, ll, st = fit_gpd_dynamic(y - tau_path(y, tp['a'], tp['b'], tp['q']))
    F = run(y, tp, th)
    xi_s, de_s = pd.Series(F['xi'], dates), pd.Series(F['delta'], dates)
    n_exc = int((F['x'] > 0).sum())
    k = 6
    aic_dyn, aic_st = 2 * k - 2 * ll, 2 * 2 - 2 * st['loglik']
    thc, llc = fit_constant_shape(y - tau_path(y, tp['a'], tp['b'], tp['q']))
    aic_const = 2 * 4 - 2 * llc

    # one-day changes of xi and delta after the event days (the filter reacts on t + 1)
    ev = {}
    for d0, lab in EVENTS.items():
        i = dates.get_loc(pd.Timestamp(d0))
        ev[d0] = dict(label=lab, loss=float(y[i]), exceed=bool(F['x'][i] > 0),
                      dxi=float(100 * (F['xi'][i + 1] / F['xi'][i] - 1)),
                      ddelta=float(100 * (F['delta'][i + 1] / F['delta'][i] - 1)))

    def chg(s, a, b):
        return float(100 * (s.loc[b:].iloc[0] / s.loc[:a].iloc[-1] - 1))

    h1 = y > F['var1']
    h25 = y > F['var25']
    dxi = xi_s.pct_change().shift(-1)        # change caused by day t
    top_xi = dxi.nlargest(1)

    # the same model on 2000-2015 only (404 exceedances): is xi_t identified?
    n_est = int((dates <= EST_END).sum())
    tpe = fit_tau(y[:n_est])
    xe = y[:n_est] - tau_path(y[:n_est], tpe['a'], tpe['b'], tpe['q'])
    the, lle, ste = fit_gpd_dynamic(xe)
    xie, _, _ = gpd_filter(xe, the)

    res = dict(
        paper=dict(sample='1962-07-03 to 2020-12-31', T=14726, T_exc=1495, a_xi=0.025, a_delta=0.112, b_xi=0.9993,
                   b_delta=0.995, a_tau=0.241, b_tau=0.989, xi_range=[0.05, 0.25], delta_range=[0.2, 2.0],
                   exc_rate_var99=0.010, mean_loss_beyond_var99=3.54, mean_es99=3.12),
        sample=dict(first=str(dates[0].date()), last=str(dates[-1].date()), T=int(len(y)), T_exc=n_exc),
        tau=tp, theta=dict(zip(NAMES, th)), loglik=ll, static=st, aic_dyn=aic_dyn, aic_static=aic_st,
        lr=2 * (ll - st['loglik']), loglik_const_shape=llc, aic_const_shape=aic_const,
        theta_const_shape=dict(zip(['fbar_xi', 'fbar_delta', 'a_delta', 'b_delta'], thc)), xi_const_shape=float(np.exp(thc[0])),
        xi_min=float(xi_s.min()), xi_max=float(xi_s.max()), xi_min_date=str(xi_s.idxmin().date()),
        xi_max_date=str(xi_s.idxmax().date()), xi_p05=float(xi_s.quantile(0.05)), xi_p95=float(xi_s.quantile(0.95)),
        delta_min=float(de_s.min()), delta_max=float(de_s.max()), delta_max_date=str(de_s.idxmax().date()),
        top_dxi_date=str(top_xi.index[0].date()), top_dxi=float(100 * top_xi.iloc[0]),
        exc_rate_var1=float(h1.mean()), n_exc_var1=int(h1.sum()), mean_loss_beyond_var1=float(y[h1].mean()),
        mean_es1=float(F['es1'].mean()), mean_es1_exc_days=float(F['es1'][h1].mean()),
        exc_rate_var25=float(h25.mean()), n_exc_var25=int(h25.sum()), mean_loss_beyond_var25=float(y[h25].mean()),
        mean_es25_exc_days=float(F['es25'][h25].mean()),
        var1_min=float(F['var1'].min()), var1_max=float(F['var1'].max()), var1_max_date=str(dates[np.argmax(F['var1'])].date()),
        xi_last=float(F['xi'][-1]), var1_last=float(F['var1'][-1]), es25_last=float(F['es25'][-1]),
        events=ev, gfc=dict(dxi=chg(xi_s, '2008-08-29', '2009-03-31'), ddelta=chg(de_s, '2008-08-29', '2009-03-31')),
        covid=dict(dxi=chg(xi_s, '2020-02-19', '2020-04-30'), ddelta=chg(de_s, '2020-02-19', '2020-04-30')),
        est2015=dict(n=n_est, n_exc=int((xe > 0).sum()), theta=dict(zip(NAMES, the)), loglik=lle, static=ste,
                     xi_median=float(np.median(xie)), xi_share_below_001=float(np.mean(xie < 0.01)),
                     xi_max=float(xie.max())),
    )
    with open('ch7_tailshape.json', 'w') as f:
        json.dump(res, f, indent=1)
    pd.DataFrame({'loss': y, 'tau': F['tau'], 'xi': F['xi'], 'delta': F['delta'], 'var1': F['var1'],
                  'es25': F['es25'], 'share': F['share']}, index=dates).rename_axis('date') \
        .to_csv('ch7_tailshape_series.csv', float_format='%.6f')
    chart(dates, y, F, st, res['paper'], res['xi_const_shape'])
    return res


def chart(dates, y, F, st, paper, xi_const):
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(5.9, 2.75), gridspec_kw=dict(width_ratios=[1.45, 1, 1]))
    a1.vlines(dates, 0, np.clip(y, 0, None), color=Teal, alpha=0.7, linewidth=0.35, label='Daily loss (gains set to 0)')
    a1.plot(dates, F['tau'], color=Forest, lw=0.6, label=r'Threshold $\tau_t$ (90% quantile)')
    a1.plot(dates, F['var1'], color=MainBlue, lw=0.8, label='VaR 1%')
    a1.plot(dates, F['es25'], color=IDAred, lw=0.6, ls=(0, (2, 1.2)), label='ES 2.5%')
    hit = y > F['var1']
    a1.scatter(dates[hit], y[hit], s=3, color=Orange, zorder=4, lw=0, label='Loss above VaR 1%')
    a1.set_ylabel('Daily loss (%)', fontsize=8)
    a1.set_title('S&P 500 losses, VaR and ES', fontsize=8.6)
    a1.set_ylim(0, 13.5)
    a2.axhspan(*paper['xi_range'], color=BandBlue, alpha=0.6, lw=0, label="Paper's range, 1962–2020")
    a2.plot(dates, F['xi'], color=MainBlue, lw=0.8, label=r'Score-driven $\xi_t$, $\delta_t$')
    a2.axhline(st['xi'], color=Navy, lw=0.7, ls='--', label='Static GPD, same thresholds')
    a2.axhline(xi_const, color=Orange, lw=0.9, ls='-.', label=r'Constant $\xi$, score-driven $\delta_t$')
    a2.set_title(r'Tail shape $\xi_t$', fontsize=8.6)
    a3.plot(dates, F['delta'], color=MainBlue, lw=0.8)
    a3.axhline(st['delta'], color=Navy, lw=0.7, ls='--')
    a3.set_title(r'Tail scale $\delta_t$ (%)', fontsize=8.6)
    for ax in (a1, a2, a3):
        ax.tick_params(labelsize=7.6)
        ax.xaxis.set_major_locator(mdates.YearLocator(10 if ax is not a1 else 5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    h1, l1 = a1.get_legend_handles_labels()
    h2, l2 = a2.get_legend_handles_labels()
    fig.legend(h1 + h2, l1 + l2, loc='lower center', ncol=3, fontsize=7.4, bbox_to_anchor=(0.5, -0.02),
               handlelength=1.2, columnspacing=0.7)
    fig.subplots_adjust(left=0.07, right=0.995, top=0.91, bottom=0.33, wspace=0.3)
    for ext in ('pdf', 'png'):
        fig.savefig(f'ch7_tailshape.{ext}', transparent=True, dpi=200)
    plt.close(fig)
    print('   saved ch7_tailshape')




## Run

In [6]:
res = main()
print(json.dumps({k: v for k, v in res.items() if k not in ("paper", "events")}, indent=1))
print(json.dumps(res["events"], indent=1))

   saved ch7_tailshape
{
 "sample": {
  "first": "2000-01-04",
  "last": "2026-09-18",
  "T": 6717,
  "T_exc": 680
 },
 "tau": {
  "a": 0.36480717435923576,
  "b": 0.98453059438647,
  "q": 1.2612061242166388,
  "loss": 0.2032692353706876
 },
 "theta": {
  "fbar_xi": -2.4886352478968274,
  "fbar_delta": -0.6033528732822879,
  "a_xi": 0.1297272355565376,
  "a_delta": 0.15952383840546672,
  "b_xi": 0.9974995432237252,
  "b_delta": 0.9859679058227908
 },
 "loglik": -380.8543125805342,
 "static": {
  "xi": 0.1987737599001438,
  "delta": 0.5629803244225507,
  "loglik": -424.4773530005309
 },
 "aic_dyn": 773.7086251610684,
 "aic_static": 852.9547060010618,
 "lr": 87.2460808399934,
 "loglik_const_shape": -381.8916037764021,
 "aic_const_shape": 771.7832075528042,
 "theta_const_shape": {
  "fbar_xi": -2.8746322495591476,
  "fbar_delta": -0.581531946277454,
  "a_delta": 0.1572364634982221,
  "b_delta": 0.9856076644468086
 },
 "xi_const_shape": 0.056436890379289,
 "xi_min": 0.01559767471470635,
 "

![ch7_tailshape](./ch7_tailshape.png)

Output: `ch7_tailshape.pdf`